# A Walk Through Combinatorics — Miklós Bóna
## Chapter 1: Section 1.1 — The Basic Pigeon-Hole Principle

---

This section was genuinely difficult for me, especially the 7, 77, 777 divisibility example. I'm writing these notes to work through it piece by piece until the whole thing clicks. I'll revisit any part that still feels murky.

---

## The Theorem

**Theorem 1.1 (Pigeon-Hole Principle).** Let $n$ and $k$ be positive integers with $n > k$. Suppose we place $n$ identical balls into $k$ identical boxes. Then there will be at least one box containing at least two balls.

### My Initial Thinking

The statement honestly sounds obvious when you read it — more balls than boxes, so some box must be crowded. But Bóna's point is that this *obvious* fact becomes extremely powerful when you frame the right things as "balls" and "boxes."

The proof is by contradiction: assume every box has at most one ball. Then the total number of balls is at most $k$ (one per box). But we said we have $n > k$ balls — contradiction. So the assumption was wrong. ∎

### The Key Mental Move

Bóna says this principle "epitomizes one of the most attractive treats of combinatorics: the possibility of obtaining very strong results by very simple means." I want to internalize that. The setup is always:

| Role | What it is in the problem |
|------|--------------------------|
| **Balls** | The objects you're distributing |
| **Boxes** | The categories / possible values |
| **Condition** | More balls than boxes → guaranteed collision |

The 7, 77, 777 example is the hardest one in this section. I'll work through it in detail below.

---

## Part 1: The Remainder Cycle — Building Intuition with Mod

Before I can understand the 7, 77, 777 proof, I need to make the idea of remainders feel natural. Here's how I think about it.

### The Circle Model

When I divide any number by $n$ and take the remainder, I always land somewhere in the set $\{0, 1, 2, \ldots, n-1\}$. That's it — only $n$ possible landing spots. I find it helpful to picture these as positions on a clock face. Every number maps to one position. As I keep adding something, I just walk around the clock.

**Example with mod 3:**

$7 \div 3$ gives remainder $1$ → position 1  
$14 \div 3$ gives remainder $2$ → position 2  
$21 \div 3$ gives remainder $0$ → position 0  
$28 \div 3$ gives remainder $1$ → back to position 1

Adding 7 each time moves me $+1$ step on the clock (since $7 \equiv 1 \pmod{3}$). The sequence of remainders cycles endlessly.

**The key insight I want to see computationally:** for any starting number and any modulus, the remainders form a repeating cycle. There are only finitely many positions on the clock — so the cycle must come back around.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from math import gcd

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = [14, 5]
plt.rcParams['font.size'] = 11

# --- Visualize the remainder cycle for mod 3, mod 7, mod 13 ---

def plot_remainder_circle(ax, modulus, steps=30, color='steelblue'):
    """Plot the sequence of remainders as positions on a circle."""
    n = modulus
    angles = [2 * np.pi * k / n for k in range(n)]
    cx = np.cos(angles)
    cy = np.sin(angles)

    # Draw circle nodes
    ax.scatter(cx, cy, s=200, color='lightgray', edgecolors='black', zorder=3)
    for k in range(n):
        ax.text(cx[k] * 1.18, cy[k] * 1.18, str(k), ha='center', va='center', fontsize=9)

    # Walk the cycle: a_i = 7*i mod n
    remainders = [(7 * i) % n for i in range(1, steps + 1)]
    prev = None
    for idx, r in enumerate(remainders):
        angle = 2 * np.pi * r / n
        x, y = np.cos(angle), np.sin(angle)
        if prev is not None:
            pangle = 2 * np.pi * prev / n
            px, py = np.cos(pangle), np.sin(pangle)
            ax.annotate('', xy=(x * 0.88, y * 0.88), xytext=(px * 0.88, py * 0.88),
                        arrowprops=dict(arrowstyle='->', color=color, lw=1.0, alpha=0.5))
        prev = r

    # Highlight first few steps
    for idx in range(min(6, len(remainders))):
        r = remainders[idx]
        angle = 2 * np.pi * r / n
        x, y = np.cos(angle), np.sin(angle)
        ax.scatter([x], [y], s=120, color=color, zorder=4, alpha=0.8)
        ax.text(x * 0.68, y * 0.68, f'$7·{idx+1}$', ha='center', va='center', fontsize=7, color=color)

    ax.set_xlim(-1.4, 1.4)
    ax.set_ylim(-1.4, 1.4)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(f'Remainder cycle mod {n}\n(walking by steps of 7)', fontsize=10)

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
colors = ['steelblue', 'darkorange', 'green']
moduli = [3, 7, 13]

for ax, mod, col in zip(axes, moduli, colors):
    plot_remainder_circle(ax, mod, steps=20, color=col)

plt.suptitle('Remainders form a cycle — only finitely many positions on the clock', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# Print the actual remainder sequences
print("Remainder sequences when computing 7, 14, 21, 28, ... mod n:\n")
for mod in moduli:
    seq = [(7 * i) % mod for i in range(1, mod + 3)]
    print(f"  mod {mod:2d}: {seq}")
print("\nNotice: the sequences always cycle back. There are only n spots on the circle.")

---

## Part 2: Subtracting Same Remainders Always Gives Zero — Why?

This is the step that confused me most. I could *see* it happening, but I couldn't explain *why* it must always work. Let me nail it down now.

### The Concrete Example (No Symbols Yet)

Take mod 3. Suppose:

- $14$ leaves remainder $2$ (because $14 = 12 + 2 = 3 \times 4 + 2$)
- $5$ leaves remainder $2$ (because $5 = 3 + 2 = 3 \times 1 + 2$)

Both numbers have the same "leftover" part ($+2$). Now subtract:

$$14 - 5 = (12 + 2) - (3 + 2)$$

The $+2$ and $-2$ literally cancel each other out:

$$= 12 - 3 = 9$$

And $9 = 3 \times 3$ — perfectly divisible by 3.

### Why the Remainder Cancels (Algebraically)

Any number $a$ with remainder $r$ when divided by $n$ can be written as:

$$a = n \cdot k + r \quad \text{("$k$ full groups of $n$, plus leftover $r$")}$$

If two numbers $a$ and $b$ have the **same** remainder $r$:

$$a = n \cdot k_1 + r$$
$$b = n \cdot k_2 + r$$

Subtract them:

$$a - b = (n \cdot k_1 + r) - (n \cdot k_2 + r) = n \cdot k_1 - n \cdot k_2 = n(k_1 - k_2)$$

The remainder $r$ cancels completely. What's left is a pure multiple of $n$. **That's why it's always divisible.**

### My Own Intuition for This

I think of it this way: two numbers with the same remainder are like two runners who've both stopped at the same point on a circular track. The distance between them is a whole number of laps — no fractional lap involved. Subtracting them just measures how many full laps apart they are.

In [ ]:
# Brute-force verify: for any two numbers with the same remainder mod n,
# their difference is always divisible by n.

def verify_same_remainder_subtraction(modulus, up_to=100):
    """Check every pair (a, b) with same remainder mod n, verify (a-b) % n == 0."""
    failures = 0
    checks = 0
    for a in range(1, up_to):
        for b in range(1, a):
            if a % modulus == b % modulus:
                diff = a - b
                if diff % modulus != 0:
                    failures += 1
                checks += 1
    return checks, failures

print("Verifying: if a ≡ b (mod n), then (a - b) is divisible by n\n")
for n in [3, 7, 13, 17]:
    checks, failures = verify_same_remainder_subtraction(n)
    status = "✅ PASS" if failures == 0 else "❌ FAIL"
    print(f"  mod {n:2d}: checked {checks:4d} pairs — {status} ({failures} failures)")

print()
print("Concrete example breakdown (mod 3):")
pairs = [(a, b) for a in range(1, 30) for b in range(1, a) if a % 3 == b % 3][:8]
for a, b in pairs:
    diff = a - b
    print(f"  {a} - {b} = {diff:3d}  |  {a}%3={a%3}, {b}%3={b%3}, {diff}%3={diff%3}")

---

## Part 3: The Sequence 7, 77, 777, ... — How It's Actually Built

I used to think of this sequence as "just appending a 7." But there's a cleaner way to see it algebraically, and it matters for the proof.

### The Recurrence

Each term is built from the previous one:

$$a_1 = 7$$
$$a_{i+1} = a_i \times 10 + 7$$

This makes sense: to go from $77$ to $777$, I shift all digits one place left (multiply by 10), then append a $7$. So:

$$77 \times 10 + 7 = 770 + 7 = 777 \checkmark$$

This is **not** the same as adding 7 each time (that gives $7, 14, 21, \ldots$). These are completely different sequences. The remainder behavior is also completely different — and that's the whole point.

### The Goal

We want to show: **at least one element of this sequence is divisible by 2003.**

The approach: instead of hunting for the divisible element directly, look at what happens to the remainders mod 2003 as we walk through the sequence. If any remainder is 0, we're done. If not — the Pigeonhole Principle forces a collision, and we exploit it.

In [ ]:
# Generate the sequence a_i = 7, 77, 777, ... using the recurrence
# and check remainders mod 2003

def generate_sevens_sequence(count):
    """Generate first `count` elements of 7, 77, 777, ..."""
    seq = []
    a = 0
    for _ in range(count):
        a = a * 10 + 7
        seq.append(a)
    return seq

# Only need remainders mod 2003 — no need to store giant numbers
def generate_remainders(count, modulus):
    remainders = []
    r = 0
    for _ in range(count):
        r = (r * 10 + 7) % modulus
        remainders.append(r)
    return remainders

mod = 2003
remainders_2003 = generate_remainders(mod, mod)

print(f"Checking remainders of 7, 77, 777, ... mod {mod}")
print(f"Total elements checked: {mod}\n")

zero_positions = [i + 1 for i, r in enumerate(remainders_2003) if r == 0]
if zero_positions:
    print(f"✅ Found elements divisible by {mod} at positions: {zero_positions[:5]}")
    pos = zero_positions[0]
    print(f"\n   The {pos}-digit repunit of 7s is divisible by {mod}!")
else:
    print(f"No element among the first {mod} was directly divisible — collision must occur.")

print(f"\nFirst 20 remainders: {remainders_2003[:20]}")
print(f"Distinct remainder values among first {mod}: {len(set(remainders_2003))}")
print(f"(There are only {mod - 1} = 2002 nonzero slots — so a collision is guaranteed by Pigeonhole)")

---

## Part 4: Subtracting Two Elements — The "7s Followed by Zeros" Structure

This is the part of the proof that really confused me visually. Let me build up to it slowly.

### Why Subtraction Gives That Shape

Let's take a concrete example. Suppose $j = 5$ and $i = 2$:

$$a_5 = 77777$$
$$a_2 = 77$$

Subtract column by column:

```
  7 7 7 7 7
-       7 7
-----------
  7 7 0 0 0
```

The rightmost $i = 2$ digits cancel (both have the same digit 7 there), leaving zeros. The remaining $j - i = 3$ leading digits stay as 7s.

**Result:** $77000$ — which is $(j-i)$ sevens followed by $i$ zeros.

### The Algebraic Identity

This pattern always holds:

$$a_j - a_i = a_{j-i} \cdot 10^i$$

Reading this in plain English: the difference equals "the shorter all-sevens number" ($a_{j-i}$) with $i$ zeros appended (multiplied by $10^i$).

The $10^i$ is literally just the "append $i$ zeros" operation. It shifts the number left by $i$ decimal places. Nothing more mysterious than that.

**This is the key factorization** — it lets us separate the problem into two parts: the "7s part" and the "zeros part."

In [ ]:
# Verify the identity: a_j - a_i = a_{j-i} * 10^i
# for several pairs (i, j) with i < j

def a(n):
    """Return the n-digit all-sevens number: 7, 77, 777, ..."""
    result = 0
    for _ in range(n):
        result = result * 10 + 7
    return result

print("Verifying: a_j - a_i = a_{j-i} * 10^i\n")
print(f"{'j':>4} {'i':>4} | {'a_j':>12} {'a_i':>12} | {'difference':>15} | {'a_{j-i} * 10^i':>18} | match?")
print("-" * 80)

test_pairs = [(3, 1), (4, 1), (4, 2), (5, 2), (5, 3), (6, 2), (7, 4)]
for j, i in test_pairs:
    diff = a(j) - a(i)
    factored = a(j - i) * (10 ** i)
    match = "✅" if diff == factored else "❌"
    print(f"{j:>4} {i:>4} | {a(j):>12} {a(i):>12} | {diff:>15} | {factored:>18} | {match}")

print()
print("Structure example — a_5 - a_2:")
print(f"  a_5 = {a(5)}")
print(f"  a_2 = {a(2)}")
print(f"  diff = {a(5) - a(2)}")
print(f"  a_3 = {a(3)}  (the '{3}-sevens' number)")
print(f"  a_3 * 10^2 = {a(3)} * {10**2} = {a(3) * 10**2}")
print(f"\n  The zeros are just: 'append i={2} zeros' → multiply by 10^{2}")

---

## Part 5: Why $10^k$ Can't "Hide" 2003 — The Factor Argument

This is the final conceptual hurdle. Once I understand why 2003 can't come from the $10^k$ part, the proof is complete.

### The Question

I know that:

$$a_{j-i} \times 10^i \quad \text{is divisible by } 2003$$

Could the divisibility by 2003 be "coming from" the $10^i$ part?

### What Does $10^i$ Contain?

$$10 = 2 \times 5$$
$$10^i = 2^i \times 5^i$$

So $10^i$ only ever contains the prime factors **2 and 5**. Nothing else. No matter how large $i$ gets, $10^i$ is built entirely from 2s and 5s.

### What About 2003?

If 2003 is prime, then its only factors are 1 and 2003. It's not divisible by 2 or 5 — so 2003 cannot appear as a factor inside $10^i$.

### The Logical Conclusion

We have:

$$(\text{7s-number}) \times 10^i \equiv 0 \pmod{2003}$$

Since 2003 and $10^i$ share no common factors ($\gcd(2003, 10^i) = 1$), the 2003 has nowhere to "hide" inside the $10^i$ piece. It must come entirely from the 7s-number.

**Therefore $a_{j-i}$ is divisible by 2003.** ∎

### Mental Model

Think of $2003$ as an object that can only fit through a door with the right shape. The $10^i$ door only opens for things made of 2s and 5s — 2003 cannot get through. The only door it can pass through is the $a_{j-i}$ door. So that's where it must be.

In [ ]:
from sympy import isprime

# Verify 2003 is prime, then confirm gcd(2003, 10^k) = 1 for all k

print("Is 2003 prime?", isprime(2003), "✅" if isprime(2003) else "❌")
print()

print("Prime factorization of 10^k:")
for k in range(1, 8):
    print(f"  10^{k} = {10**k}  →  factors: only 2s and 5s")

print()
print("gcd(2003, 10^k) for k = 1 to 10:")
for k in range(1, 11):
    g = gcd(2003, 10**k)
    print(f"  gcd(2003, 10^{k:2d}) = {g}  →  {'share NO factors ✅' if g == 1 else 'share factors ❌'}")

print()
print("Conclusion: 2003 and any power of 10 are coprime (share no factors).")
print("So if (7s-number) × 10^k is divisible by 2003,")
print("then the 7s-number itself must be divisible by 2003.")

---

## Part 6: Putting It All Together — The Full Proof, Step by Step

Now I want to walk through the complete argument as one connected story, with code showing each step happening live. This is the version I'll come back to when I want to remind myself how the proof works.

### The Proof in Plain English

1. Look at the first 2003 elements of $7, 77, 777, \ldots$
2. Each has a remainder mod 2003 between 0 and 2002.
3. **If any remainder is 0**, we're done — that element is directly divisible by 2003.
4. **If no remainder is 0**, all 2003 remainders fall into only 2002 possible nonzero values.
5. By Pigeonhole: two elements must share a remainder. Call them $a_i$ and $a_j$ ($i < j$).
6. Their difference $a_j - a_i$ is divisible by 2003 (from Part 2).
7. That difference equals $a_{j-i} \times 10^i$ (from Part 4).
8. Since $\gcd(2003, 10^i) = 1$ (from Part 5), the 7s-number $a_{j-i}$ is divisible by 2003. ∎

The mapping to Pigeonhole:
- **Balls** = the first 2003 elements of the sequence
- **Boxes** = the 2002 nonzero remainder values $\{1, 2, \ldots, 2002\}$
- **More balls than boxes** → collision guaranteed

In [ ]:
# Full interactive proof walkthrough for mod = 2003

def full_proof_walkthrough(modulus):
    print(f"{'='*60}")
    print(f"PROOF WALKTHROUGH  (mod = {modulus})")
    print(f"{'='*60}\n")

    # Step 1: Generate remainders
    remainders = []
    r = 0
    for i in range(1, modulus + 1):
        r = (r * 10 + 7) % modulus
        remainders.append(r)

    print(f"Step 1 — Generate remainders of a_1, a_2, ..., a_{modulus} mod {modulus}")
    print(f"         First 10 remainders: {remainders[:10]}\n")

    # Step 2: Check for direct zero
    zeros = [(i + 1, r) for i, r in enumerate(remainders) if r == 0]
    if zeros:
        first_zero_pos = zeros[0][0]
        print(f"Step 2 — Direct hit! a_{first_zero_pos} has remainder 0 → directly divisible by {modulus}!")
        print(f"         (All zero positions among first {modulus}: {[z[0] for z in zeros[:5]]})\n")
    else:
        print(f"Step 2 — No direct zero found among first {modulus} elements.")
        print(f"         All remainders are between 1 and {modulus - 1}.\n")

    # Step 3: Find first collision
    seen = {}
    collision_i, collision_j = None, None
    for idx, rem in enumerate(remainders):
        i_1indexed = idx + 1
        if rem in seen:
            collision_j = i_1indexed
            collision_i = seen[rem]
            break
        seen[rem] = i_1indexed

    if collision_i and collision_j:
        shared_rem = remainders[collision_i - 1]
        print(f"Step 3 — First remainder collision:")
        print(f"         a_{collision_i} and a_{collision_j} both have remainder {shared_rem} mod {modulus}\n")

        # Step 4: Compute difference structure
        length = collision_j - collision_i
        print(f"Step 4 — Difference structure:")
        print(f"         a_{collision_j} - a_{collision_i}  =  a_{length} × 10^{collision_i}")
        print(f"         (That's {length} sevens followed by {collision_i} zeros)\n")

        # Step 5: Verify divisibility
        # Work mod modulus to avoid huge numbers
        a_length_mod = 0
        for _ in range(length):
            a_length_mod = (a_length_mod * 10 + 7) % modulus
        pow10_mod = pow(10, collision_i, modulus)
        product_mod = (a_length_mod * pow10_mod) % modulus

        print(f"Step 5 — Verify a_{length} × 10^{collision_i} ≡ 0 (mod {modulus}):")
        print(f"         (a_{length} mod {modulus}) × (10^{collision_i} mod {modulus}) mod {modulus} = {product_mod}")
        print(f"         → Divisible by {modulus}? {'✅ Yes' if product_mod == 0 else '❌ No'}\n")

        print(f"Step 6 — Factor argument:")
        print(f"         gcd({modulus}, 10^{collision_i}) = {gcd(modulus, 10**collision_i)}")
        print(f"         Since gcd = 1, the 10^{collision_i} part carries no factor of {modulus}.")
        print(f"         → Therefore a_{length} itself is divisible by {modulus}!")
        print(f"\n         a_{length} mod {modulus} = {a_length_mod}  {'✅' if a_length_mod == 0 else '❌'}")

full_proof_walkthrough(2003)

---

## Part 7: Verify Numerically — Small Cases Where I Can See Everything

The 2003 case involves giant numbers I can't inspect by hand. But the exact same argument works for small moduli like 3, 7, and 13 — and there I can see every number directly. Running those cases first is how I convinced myself the proof is correct.

### Why This Helps

If I can trace the full argument with $n = 3$:
- generate $7, 77, 777$
- watch the remainder collision happen
- do the subtraction and see the 7s-followed-by-zeros shape
- check that the smaller all-7s number is divisible

…then I know I truly understand it, not just pattern-matched on symbols.

In [ ]:
# Small-case walkthroughs where I can inspect actual numbers

def small_case_walkthrough(modulus):
    print(f"\n{'─'*55}")
    print(f"  SMALL CASE: mod = {modulus}")
    print(f"{'─'*55}")

    # Generate actual numbers (small enough to print)
    elements = []
    v = 0
    for i in range(1, modulus + 2):
        v = v * 10 + 7
        elements.append(v)

    # Print the sequence with remainders
    print(f"\n  i  |  a_i{'':>15}|  a_i mod {modulus}")
    print(f"  {'-'*40}")
    for i, val in enumerate(elements[:modulus], start=1):
        rem = val % modulus
        marker = " ← 0 ✅" if rem == 0 else ""
        print(f"  {i:2d} |  {val:>18} |  {rem}{marker}")

    # Find first collision or zero
    seen = {}
    for i, val in enumerate(elements[:modulus], start=1):
        rem = val % modulus
        if rem == 0:
            print(f"\n  ✅ Direct hit: a_{i} = {val} is divisible by {modulus}!")
            print(f"     {val} ÷ {modulus} = {val // modulus} remainder {val % modulus}")
            return
        if rem in seen:
            j = i
            i_prev = seen[rem]
            break
        seen[rem] = i
    else:
        return

    # Show the collision and subtraction
    a_i = elements[i_prev - 1]
    a_j = elements[j - 1]
    diff = a_j - a_i
    length = j - i_prev
    a_short = elements[length - 1]
    power = 10 ** i_prev

    print(f"\n  Remainder collision:  a_{i_prev} and a_{j} both ≡ {elements[i_prev-1] % modulus} (mod {modulus})")
    print(f"\n  Subtraction:")
    print(f"    a_{j} = {a_j}")
    print(f"  - a_{i_prev} = {a_i}")
    print(f"  {'─'*30}")
    print(f"    diff = {diff}  ← {length} sevens, then {i_prev} zeros")
    print(f"\n  Factor out 10^{i_prev} = {power}:")
    print(f"    {diff} = {a_short} × {power}")
    print(f"    a_{length} = {a_short}")
    print(f"\n  gcd({modulus}, 10^{i_prev}) = gcd({modulus}, {power}) = {gcd(modulus, power)}")
    print(f"  a_{length} mod {modulus} = {a_short % modulus}  → {'divisible ✅' if a_short % modulus == 0 else 'NOT divisible ❌'}")

for mod in [3, 7, 13]:
    small_case_walkthrough(mod)

In [ ]:
# Visual summary: plot the remainder sequences for mod 3, 7, 13
# and mark where the first collision occurs

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors = ['steelblue', 'darkorange', 'green']

for ax, mod, col in zip(axes, [3, 7, 13], colors):
    rems = []
    r = 0
    seen = {}
    collision_idx = None
    prev_idx = None
    for i in range(1, mod + 1):
        r = (r * 10 + 7) % mod
        rems.append(r)
        if r == 0 and collision_idx is None:
            collision_idx = i
            prev_idx = None
            break
        if r in seen and collision_idx is None:
            collision_idx = i
            prev_idx = seen[r]
        elif r not in seen:
            seen[r] = i

    x = list(range(1, len(rems) + 1))
    ax.scatter(x, rems, color=col, s=60, zorder=3)
    ax.plot(x, rems, color=col, alpha=0.4, linewidth=1)
    ax.axhline(0, color='red', linestyle='--', linewidth=1, alpha=0.5, label='remainder = 0')

    if collision_idx is not None and prev_idx is not None:
        ax.axvline(collision_idx, color='purple', linestyle=':', linewidth=1.5, label=f'collision at i={collision_idx}')
        ax.axvline(prev_idx, color='purple', linestyle=':', linewidth=1.5, alpha=0.5)
        ax.scatter([collision_idx, prev_idx],
                   [rems[collision_idx - 1], rems[prev_idx - 1]],
                   color='purple', s=120, zorder=5, label='matching pair')
    elif collision_idx is not None:
        ax.axvline(collision_idx, color='red', linestyle='-', linewidth=2, label=f'zero at i={collision_idx}')
        ax.scatter([collision_idx], [0], color='red', s=150, zorder=5)

    ax.set_xlabel('Position $i$')
    ax.set_ylabel(f'$a_i \\mod {mod}$')
    ax.set_title(f'Remainders mod {mod}', fontsize=11)
    ax.set_ylim(-0.5, mod - 0.5)
    ax.legend(fontsize=8)

plt.suptitle('Remainder sequences: collision forces a divisible element', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

---

## Summary — What I Learned From This Example

This example took me a long time to click, but once it did the whole structure became clear. Here's what I'll take away:

### The Proof Strategy (Reusable Everywhere)

| Step | What I did | Why it worked |
|------|-----------|---------------|
| 1 | Looked at remainders instead of divisibility directly | Turns an infinite search into a finite one |
| 2 | Counted: 2003 numbers, only 2002 nonzero remainder slots | Sets up the Pigeonhole condition |
| 3 | Applied Pigeonhole → two numbers share a remainder | 2003 > 2002 guarantees a collision |
| 4 | Subtracted the collision pair | Remainder cancels → difference is divisible |
| 5 | Noticed the difference has a special structure: $a_{j-i} \times 10^i$ | The "7s followed by zeros" shape |
| 6 | Used $\gcd(2003, 10^i) = 1$ | Transferred divisibility to the 7s-number |

### The Big Lesson

The Pigeonhole Principle is most powerful when you identify the right "boxes." Here the boxes were **remainder values**, not anything geometric or physical. The problem of proving divisibility became a counting problem — and counting is something I can always do.

### What I Still Want to Revisit

- The formal proof that $a_j - a_i = a_{j-i} \times 10^i$ (I verified it computationally but haven't written an algebraic proof yet)
- More examples from Section 1.1 — I want to see what other "balls" and "boxes" look like in other problems
- The generalized Pigeonhole Principle ($n$ balls, $k$ boxes → at least one box has $\lceil n/k \rceil$ balls)